### Cell 1 — Imports and Configuration

In [13]:
import pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

FILES_PATH = "/lakehouse/default/Files"
FILE_PATH = "abfss://Swift@onelake.dfs.fabric.microsoft.com/lh_Bronze_StratusCoreTelecoms.Lakehouse/Files/StratusCore Telecoms.xlsx"
RUN_TS     = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"✅ Config loaded — Run timestamp: {RUN_TS}")
print(f"📁 Source file: {FILE_PATH}")

StatementMeta(, 0a108e28-3942-4a76-a0f6-931a1261bd44, 20, Finished, Available, Finished, False)

✅ Config loaded — Run timestamp: 2026-04-17 13:03:23
📁 Source file: abfss://Swift@onelake.dfs.fabric.microsoft.com/lh_Bronze_StratusCoreTelecoms.Lakehouse/Files/StratusCore Telecoms.xlsx


### Cell 2 — Helper: Clean Column Names

In [14]:
def clean_col_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardise column names:
      - Strip leading/trailing whitespace
      - Replace spaces with underscores
      - Remove all characters except alphanumerics and underscores
      - Cast everything to string (Bronze stores raw strings only)
    """
    df.columns = (
        df.columns
          .astype(str)
          .str.strip()
          .str.replace(r'\s+', '_', regex=True)
          .str.replace(r'[^a-zA-Z0-9_]', '', regex=True)
    )
    return df.astype(str)

StatementMeta(, 0a108e28-3942-4a76-a0f6-931a1261bd44, 21, Finished, Available, Finished, False)

### Cell 3 — Read Outage Sheet

In [15]:
print("📖 Reading Outage sheet...")
df_outage_raw = pd.read_excel(
    FILE_PATH,
    sheet_name="Outage",
    dtype=str           # Read everything as string — Bronze never assumes types
)
print(f"   Shape: {df_outage_raw.shape[0]} rows × {df_outage_raw.shape[1]} columns")
print(f"   Columns: {list(df_outage_raw.columns)}")
df_outage_raw.head(3)

StatementMeta(, 0a108e28-3942-4a76-a0f6-931a1261bd44, 22, Finished, Available, Finished, False)

📖 Reading Outage sheet...
   Shape: 200 rows × 11 columns
   Columns: ['ID', 'Date ', 'Severity', 'Company', 'Region', 'BSP', 'Outage Start', 'Outage End', 'Actual Event Down Time', 'Outage in Secs', 'Type of Fault']


,ID,Date,Severity,Company,Region,BSP,Outage Start,Outage End,Actual Event Down Time,Outage in Secs,Type of Fault
0,1,2925-10-07 00:00:00,Major,NovaTel,North,BSP-C,2925-10-07 02:45:00,2925-10-07 07:21:36,04:36:36,16596,Fiber Cut
1,2,2925-08-03 00:00:00,Minor,SkyGrid,Central,BSP-A,2025-08-03 07:45:00,2025-08-03 10:16:48,02:31:48,9108,Link Down
2,3,2925-06-22 00:00:00,Major,NovaTel,South,BSP-D,2025-06-22 03:00:00,2025-06-22 10:42:00,07:42:00,27720,Equipment Failure


### Cell 4 — Read Availability Sheet (header=None is critical)

In [16]:
print("📖 Reading Availability sheet...")
# IMPORTANT: header=None preserves the two-row header structure.
# If we let pandas guess the header it will collapse rows 0 and 1 into
# a single multi-level index and destroy the FM category groupings.
df_avail_raw = pd.read_excel(
    FILE_PATH,
    sheet_name="Availability",
    header=None,
    dtype=str
)
print(f"   Shape: {df_avail_raw.shape[0]} rows × {df_avail_raw.shape[1]} columns")
print(f"\n   First 3 rows (raw):")
print(df_avail_raw.iloc[:3].to_string())

StatementMeta(, 0a108e28-3942-4a76-a0f6-931a1261bd44, 24, Finished, Available, Finished, False)

📖 Reading Availability sheet...
   Shape: 202 rows × 29 columns

   First 3 rows (raw):
    0              1         2       3      4                    5                    6                    7                    8                    9                    10                   11                   12                   13                   14                   15                   16                   17                   18                   19                   20                   21                   22                   23                   24                   25                   26                   27                   28
0  NaN            NaN       NaN     NaN    NaN         Including FM                  NaN                  NaN                  NaN                  NaN                  NaN                  NaN                  NaN                  NaN                  NaN                  NaN                  NaN         Excluding FM                  NaN                  NaN

### Cell 5 — Save Both Sheets as Bronze Delta Tables

In [17]:
ingestion_log = []

for df_raw, table_name in [
    (df_outage_raw,  "bronze_stratus_outage"),
    (df_avail_raw,   "bronze_stratus_availability"),
]:
    df_clean = clean_col_names(df_raw.copy())

    # Add pipeline metadata columns to every Bronze table
    df_clean["_bronze_ingested_at"] = RUN_TS
    df_clean["_source_file"]        = Path(FILE_PATH).name

    row_count = len(df_clean)

    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    (
        spark.createDataFrame(df_clean)
             .write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(table_name)
    )

    ingestion_log.append({"table": table_name, "rows": row_count, "status": "✅ OK"})
    print(f"✅ Saved: {table_name}  ({row_count} rows)")

print("\n📊 Ingestion Summary:")
for entry in ingestion_log:
    print(f"   {entry['status']}  {entry['table']}  — {entry['rows']} rows")

StatementMeta(, 0a108e28-3942-4a76-a0f6-931a1261bd44, 25, Finished, Available, Finished, False)

✅ Saved: bronze_stratus_outage  (200 rows)
✅ Saved: bronze_stratus_availability  (202 rows)

📊 Ingestion Summary:
   ✅ OK  bronze_stratus_outage  — 200 rows
   ✅ OK  bronze_stratus_availability  — 202 rows


### Cell 6 — Quick Validation

In [18]:
print("🔍 Validation: Row counts from Delta tables")
for t in ["bronze_stratus_outage", "bronze_stratus_availability"]:
    count = spark.table(t).count()
    print(f"   {t}: {count} rows")

print("\n🔍 Outage — sample values (first 5 rows):")
spark.table("bronze_stratus_outage").show(5, truncate=False)

StatementMeta(, 0a108e28-3942-4a76-a0f6-931a1261bd44, 26, Finished, Available, Finished, True)

🔍 Validation: Row counts from Delta tables
   bronze_stratus_outage: 200 rows
   bronze_stratus_availability: 202 rows

🔍 Outage — sample values (first 5 rows):
+---+-------------------+--------+--------+-------+-----+-------------------+-------------------+----------------------+--------------+---------------------+-------------------+-------------------------+
|ID |Date               |Severity|Company |Region |BSP  |Outage_Start       |Outage_End         |Actual_Event_Down_Time|Outage_in_Secs|Type_of_Fault        |_bronze_ingested_at|_source_file             |
+---+-------------------+--------+--------+-------+-----+-------------------+-------------------+----------------------+--------------+---------------------+-------------------+-------------------------+
|151|2025-12-23 00:00:00|Major   |NovaTel |East   |BSP-B|2025-12-23 04:45:00|2025-12-23 13:23:24|08:38:24              |31104         |Power Failure        |2026-04-17 13:03:23|StratusCore Telecoms.xlsx|
|152|2925-11-08 00:00:0